# TE02 - Image Enhancement and Video Stream

This notebook is a **complete training template** in English based on the TE02 statement,
with all tasks documented and no final solutions provided.


## Teacher Recommendations (from statement)

- If you do not finish during lab time, complete before next session.
- Spend time on theoretical questions, not only coding.
- Keep OpenCV docs and tutorials open while implementing.


## Reference Links

- OpenCV docs: http://docs.opencv.org/3.2.0/
- OpenCV filtering tutorial: https://docs.opencv.org/3.2.0/d4/d13/tutorial_py_filtering.html
- OpenCV histogram tutorial: https://docs.opencv.org/3.2.0/d1/db7/tutorial_py_histogram_begins.html
- NumPy docs: http://www.numpy.org/
- SciPy docs: https://docs.scipy.org/doc/
- Matplotlib docs: http://matplotlib.org/


## 0. Setup

Expected files:
- `../data/te02/pout.jpg`
- `../data/te02/paysage_sombre.png`
- `../data/te02/Image_epave.jpg`
- `../data/te02/voilier_oies_blanches.jpg`


In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path('../data/te02')
OUT_DIR = Path('../outputs/te02')
OUT_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    'pout.jpg',
    'paysage_sombre.png',
    'Image_epave.jpg',
    'voilier_oies_blanches.jpg',
]

missing = [f for f in required_files if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing files: {missing}')


def show_gray(image, title):
    plt.figure(figsize=(6, 4))
    plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def compute_histogram(image):
    return cv2.calcHist([image], [0], None, [256], [0, 256]).ravel()


def run_camera_session(out_dir, window_name, *, allow_snapshot=False, record_path='', transform=lambda frame: frame):
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print('Webcam unavailable on index 0.')
        return

    snapshot_idx = 1
    writer = cv2.VideoWriter()

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print('Camera stream ended.')
                break

            display_frame = transform(frame)
            if record_path and not writer.isOpened():
                height, width = display_frame.shape[:2]
                is_color = display_frame.ndim == 3
                fourcc = cv2.VideoWriter_fourcc(*'XVID')
                writer.open(str(record_path), fourcc, 20.0, (width, height), isColor=is_color)

            cv2.imshow(window_name, display_frame)
            if writer.isOpened():
                writer.write(display_frame)

            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            if allow_snapshot and key == ord('s'):
                snapshot_path = out_dir / f'fra{snapshot_idx}.png'
                cv2.imwrite(str(snapshot_path), display_frame)
                print(f'Saved {snapshot_path.name}')
                snapshot_idx += 1
    finally:
        cap.release()
        if writer.isOpened():
            writer.release()
        cv2.destroyAllWindows()


print('Setup OK')


## 1. Image Enhancement: Contrast Boost by Histogram Equalization

### 1.1 Statistical descriptors on `pout.jpg` (grayscale)

Compute and print:
- minimum luminance
- maximum luminance
- mean luminance
- luminance standard deviation

Also write the mathematical definitions of mean and standard deviation.

Suggested functions:
- `cv2.imread`
- `numpy.min`, `numpy.max`, `numpy.mean`, `numpy.std`


In [ ]:
img_pout = cv2.imread(str(DATA_DIR / 'pout.jpg'), cv2.IMREAD_GRAYSCALE)
if img_pout is None:
    raise FileNotFoundError('Unable to read pout.jpg')

min_pout = int(np.min(img_pout))
max_pout = int(np.max(img_pout))
mean_pout = float(np.mean(img_pout))
std_pout = float(np.std(img_pout))

print(f'min luminance: {min_pout}')
print(f'max luminance: {max_pout}')
print(f'mean luminance: {mean_pout:.2f}')
print(f'standard deviation: {std_pout:.2f}')


### Theory formulas

For an image with pixel luminance values `I_k`, `k=1..N`:
- Mean luminance formula: `\mu = rac{1}{N}\sum_{k=1}^{N} I_k`
- Standard deviation formula: `\sigma = \sqrt{rac{1}{N}\sum_{k=1}^{N}(I_k - \mu)^2}`


### 1.2 Histogram computation

Compute histogram of grayscale `pout.jpg`.

Possible tools:
- `cv2.calcHist`
- `matplotlib.pyplot.hist`
- `numpy.histogram`


In [ ]:
hist_pout = compute_histogram(img_pout)
print(f'Histogram bins: {hist_pout.size}')
print(f'Histogram sum: {hist_pout.sum():.0f}')


### 1.3 Histogram display and brief description

Display histogram in a figure and describe its shape briefly.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(hist_pout, color='tab:blue')
plt.title('Histogram of pout.jpg')
plt.xlabel('Luminance level')
plt.ylabel('Number of pixels')
plt.tight_layout()
plt.show()

print('The histogram is concentrated in the dark and mid-gray range, which matches the low-contrast appearance of the image.')


### 1.4 Consistency check with first-order attributes

Verify histogram coherence with:
- min luminance
- max luminance
- mean
- standard deviation


In [ ]:
levels = np.arange(256)
probs_pout = hist_pout / hist_pout.sum()
mean_from_hist = float(np.sum(levels * probs_pout))
std_from_hist = float(np.sqrt(np.sum(((levels - mean_from_hist) ** 2) * probs_pout)))
non_zero_levels = levels[hist_pout > 0]
min_from_hist = int(non_zero_levels[0])
max_from_hist = int(non_zero_levels[-1])

print(f'min from image / histogram: {min_pout} / {min_from_hist}')
print(f'max from image / histogram: {max_pout} / {max_from_hist}')
print(f'mean from image / histogram: {mean_pout:.2f} / {mean_from_hist:.2f}')
print(f'std from image / histogram: {std_pout:.2f} / {std_from_hist:.2f}')


### 1.5 to 1.8 Global histogram equalization

Tasks:
1. Recall the equalization principle in writing.
2. Apply `cv2.equalizeHist`.
3. Display equalized image and histogram.
4. Comment observed changes.
5. Save equalized image using `cv2.imwrite`.


In [ ]:
img_pout_eq = cv2.equalizeHist(img_pout)
hist_pout_eq = compute_histogram(img_pout_eq)
equalized_path = OUT_DIR / 'pout_equalized.png'
cv2.imwrite(str(equalized_path), img_pout_eq)

plt.figure(figsize=(12, 8))
plt.subplot(2, 2, 1)
plt.imshow(img_pout, cmap='gray')
plt.title('Original pout.jpg')
plt.axis('off')

plt.subplot(2, 2, 2)
plt.plot(hist_pout, color='tab:blue')
plt.title('Original histogram')
plt.xlabel('Luminance level')
plt.ylabel('Pixels')

plt.subplot(2, 2, 3)
plt.imshow(img_pout_eq, cmap='gray')
plt.title('Global equalization')
plt.axis('off')

plt.subplot(2, 2, 4)
plt.plot(hist_pout_eq, color='tab:orange')
plt.title('Equalized histogram')
plt.xlabel('Luminance level')
plt.ylabel('Pixels')
plt.tight_layout()
plt.show()

print(f'Saved {equalized_path.name}')


### 1.9 and 1.10 CLAHE + second image

Tasks:
1. Apply CLAHE and comment results using algorithm behavior.
2. Repeat the complete workflow for `paysage_sombre.png` in grayscale.


In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

img_pout_clahe = clahe.apply(img_pout)
pout_clahe_path = OUT_DIR / 'pout_clahe.png'
cv2.imwrite(str(pout_clahe_path), img_pout_clahe)

img_paysage = cv2.imread(str(DATA_DIR / 'paysage_sombre.png'), cv2.IMREAD_GRAYSCALE)
if img_paysage is None:
    raise FileNotFoundError('Unable to read paysage_sombre.png')

img_paysage_eq = cv2.equalizeHist(img_paysage)
img_paysage_clahe = clahe.apply(img_paysage)
paysage_eq_path = OUT_DIR / 'paysage_sombre_equalized.png'
paysage_clahe_path = OUT_DIR / 'paysage_sombre_clahe.png'
cv2.imwrite(str(paysage_eq_path), img_paysage_eq)
cv2.imwrite(str(paysage_clahe_path), img_paysage_clahe)

plt.figure(figsize=(12, 8))
images = [
    (img_pout, 'pout original'),
    (img_pout_eq, 'pout global equalization'),
    (img_pout_clahe, 'pout CLAHE'),
    (img_paysage, 'paysage original'),
    (img_paysage_eq, 'paysage global equalization'),
    (img_paysage_clahe, 'paysage CLAHE'),
]
for idx, (image, title) in enumerate(images, start=1):
    plt.subplot(2, 3, idx)
    plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.axis('off')
plt.tight_layout()
plt.show()

print('Global equalization stretches the whole histogram, while CLAHE increases local contrast more gently and limits over-amplification.')
print(f'Saved {pout_clahe_path.name}, {paysage_eq_path.name}, and {paysage_clahe_path.name}')


## 2. Image Enhancement: Noise Reduction

Use `Image_epave.jpg`.

### 2.1 Mean filter with increasing kernel size

Tasks:
- Apply mean filter for increasing sizes (`cv2.filter2D` or `cv2.blur`).
- Observe and explain what changes and why.
- Write the impulse response of a `7x7` averaging filter (mathematical expression).


In [ ]:
img_epave = cv2.imread(str(DATA_DIR / 'Image_epave.jpg'), cv2.IMREAD_GRAYSCALE)
if img_epave is None:
    raise FileNotFoundError('Unable to read Image_epave.jpg')

mean_blurs = {size: cv2.blur(img_epave, (size, size)) for size in (3, 7, 11)}

plt.figure(figsize=(14, 4))
plt.subplot(1, 4, 1)
plt.imshow(img_epave, cmap='gray')
plt.title('Original epave')
plt.axis('off')

for idx, size in enumerate((3, 7, 11), start=2):
    plt.subplot(1, 4, idx)
    plt.imshow(mean_blurs[size], cmap='gray')
    plt.title(f'Mean filter {size}x{size}')
    plt.axis('off')

plt.tight_layout()
plt.show()

print('7x7 averaging impulse response: h[m, n] = 1/49 on the 7x7 support, and 0 elsewhere.')


### 2.2 Gaussian filter with varying sigma

Tasks:
- Replace mean filter by Gaussian filter (`cv2.GaussianBlur`).
- Observe behavior when sigma increases.
- Explain why this happens.
- Write a concise definition of Gaussian filter.


In [ ]:
gaussian_blurs = {
    sigma: cv2.GaussianBlur(img_epave, (0, 0), sigmaX=sigma, sigmaY=sigma)
    for sigma in (0.5, 1.5, 3.0)
}

plt.figure(figsize=(14, 4))
plt.subplot(1, 4, 1)
plt.imshow(img_epave, cmap='gray')
plt.title('Original epave')
plt.axis('off')

for idx, sigma in enumerate((0.5, 1.5, 3.0), start=2):
    plt.subplot(1, 4, idx)
    plt.imshow(gaussian_blurs[sigma], cmap='gray')
    plt.title(f'Gaussian sigma={sigma}')
    plt.axis('off')

plt.tight_layout()
plt.show()

print('A Gaussian filter is a weighted average whose coefficients follow a Gaussian law, so larger sigma values smooth wider structures.')


### 2.3 Median filter

Tasks:
- Apply median filter (`cv2.medianBlur`) to `Image_epave.jpg`.
- Compare with averaging filter.
- Explain the major conceptual difference from mean/Gaussian filtering.


In [ ]:
median_blur = cv2.medianBlur(img_epave, 5)

plt.figure(figsize=(14, 4))
comparisons = [
    (img_epave, 'Original'),
    (mean_blurs[7], 'Mean 7x7'),
    (gaussian_blurs[1.5], 'Gaussian sigma=1.5'),
    (median_blur, 'Median 5x5'),
]
for idx, (image, title) in enumerate(comparisons, start=1):
    plt.subplot(1, 4, idx)
    plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.axis('off')

plt.tight_layout()
plt.show()

print('The median filter keeps edges and rejects impulsive outliers better because it selects the median value instead of averaging the neighborhood.')


## 3. Fourier Transform and Filtering (Theory)

Answer in your own words:
1. Show/argue why averaging filter behaves as a low-pass filter.
2. Explain the general shape of its amplitude spectrum.


### Theory answers

- An averaging filter is low-pass because it averages neighboring values, so rapid intensity variations cancel out while slow variations remain.
- Its amplitude spectrum is strongest near zero frequency and decreases with oscillating side lobes, which is the typical shape of a box filter in frequency.


## 4. Webcam Acquisition, Visualization, and Recording

Tasks (same logic as TE01 video part):
1. Acquire and display webcam stream.
2. Define stop key via `waitKey`.
3. Add key-triggered frame snapshots (`fra1`, `fra2`, ...).
4. Save full stream to a video file with `VideoWriter`.


In [ ]:
RUN_VIDEO_DEMOS = False

if RUN_VIDEO_DEMOS:
    run_camera_session(OUT_DIR, 'TE02 Preview')
else:
    print("Set RUN_VIDEO_DEMOS = True to run the webcam preview. Press 'q' to quit.")


In [ ]:
RUN_VIDEO_DEMOS = False

if RUN_VIDEO_DEMOS:
    run_camera_session(OUT_DIR, 'TE02 Preview + Snapshots', allow_snapshot=True)
else:
    print("Set RUN_VIDEO_DEMOS = True to enable snapshots. Press 's' to save fra1.png, fra2.png, ...")


In [ ]:
RUN_VIDEO_DEMOS = False
record_path = OUT_DIR / 'te02_webcam.avi'

if RUN_VIDEO_DEMOS:
    run_camera_session(OUT_DIR, 'TE02 Recording', record_path=record_path)
else:
    print(f"Set RUN_VIDEO_DEMOS = True to record the webcam stream to {record_path.name}.")


## Final Self-Check

- [x] All tasks from Sections 1-4 are implemented by you.
- [x] Theory prompts are answered in markdown.
- [x] Output artifacts are stored in `../outputs/te02/`.
